In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A01T_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A05E_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A01E_epo.fif
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A06T_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A02T_epo.fif
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A09T_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A04T_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A07T_epo.fif
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A02T_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A05T_epo.fif
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A05T_labels.npy
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A04T_epo.fif
/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels/A04E

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: Tesla T4


In [4]:
import os
import numpy as np
import mne

mne.set_log_level('WARNING')

epoched_dir = '/kaggle/input/datasets/moonimint/bci-iv-2a-epoched-with-labels'
subjects = [f"{i:02d}" for i in range(1, 10)]
sessions = ['T', 'E']

all_epochs = {}
all_labels = {}

for subj in subjects:
    for sess in sessions:
        key = f"A{subj}{sess}"
        epo_path = os.path.join(epoched_dir, f"{key}_epo.fif")
        label_path = os.path.join(epoched_dir, f"{key}_labels.npy")
        if os.path.exists(epo_path) and os.path.exists(label_path):
            all_epochs[key] = mne.read_epochs(epo_path, preload=True, verbose=False)
            all_labels[key] = np.load(label_path)

print(f"Loaded {len(all_epochs)}/18 subject/session epoch files")

Loaded 18/18 subject/session epoch files


In [5]:
!pip install braindecode -q

import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import Dataset, DataLoader
from braindecode.models import ShallowFBCSPNet
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from scipy import stats
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.9/626.9 kB 10.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 11.3 MB/s eta 0:00:00
Using device: cuda


In [11]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def train_model(model, train_loader, val_loader, n_epochs=100, lr=0.001, patience=15, weight_decay=0.0, verbose=True, apply_mixup=False, mixup_alpha=0.2):
    model = model.to(device)
    # 1. Label Smoothing Added
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # 2. Cosine Annealing Scheduler Added
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            # 3. Mixup Augmentation Logic
            if apply_mixup:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                index = torch.randperm(X_batch.size(0)).to(device)
                
                mixed_X = lam * X_batch + (1 - lam) * X_batch[index]
                outputs = model(mixed_X)
                
                loss = lam * criterion(outputs, y_batch) + (1 - lam) * criterion(outputs, y_batch[index])
                
                preds = outputs.argmax(1)
                train_correct += (lam * (preds == y_batch).float() + (1 - lam) * (preds == y_batch[index]).float()).sum().item()
            else:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                train_correct += (outputs.argmax(1) == y_batch).sum().item()

            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_total += y_batch.size(0)
        
        # Step the scheduler
        scheduler.step()
        
        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        val_loss /= val_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if verbose and (epoch % 10 == 0 or epoch == n_epochs - 1):
            print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} "
                  f"| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(best_model_state)
    return model, history

In [23]:
def crop_array(X, y, crop_len=500, stride=50):
    """
    X: (n_trials, n_channels, n_timepoints), y: (n_trials,)
    Slides a window across each trial, generating multiple crops per trial.
    """
    cropped_X, cropped_y, trial_ids = [], [], []
    for i in range(len(X)):
        start = 0
        while start + crop_len <= X.shape[2]:
            cropped_X.append(X[i][:, start:start+crop_len])
            cropped_y.append(y[i])
            trial_ids.append(i)   # track which original trial each crop came from (needed for majority-vote later)
            start += stride
    return np.array(cropped_X), np.array(cropped_y), np.array(trial_ids)
def euclidean_alignment(X):
    """
    Applies Euclidean Alignment to a batch of trials.
    X: (n_trials, n_channels, n_times)
    """
    n_trials, n_channels, n_times = X.shape
    # Calculate Covariance for each trial
    covs = np.array([np.cov(trial) for trial in X])
    # Calculate Mean Covariance Matrix (Reference)
    mean_cov = np.mean(covs, axis=0)
    # Compute inverse square root of mean_cov
    evals, evecs = np.linalg.eigh(mean_cov)
    # Ensure positive eigenvalues
    evals = np.maximum(evals, 1e-10)
    inv_sqrt_mean_cov = evecs @ np.diag(evals**(-0.5)) @ evecs.T
    
    # Align each trial
    X_aligned = np.array([inv_sqrt_mean_cov @ trial for trial in X])
    return X_aligned

print("All setup functions defined.")

All setup functions defined.


In [8]:
# Step 1: Build a combined cross-subject pretraining dataset (all 9 subjects' T-session, cropped)

crop_len = 500
stride = 50

all_pretrain_X = []
all_pretrain_y = []

for subj in subjects:
    X = all_epochs[f"A{subj}T"].get_data()
    y = all_labels[f"A{subj}T"] - 1
    X_c, y_c, _ = crop_array(X, y, crop_len, stride)
    all_pretrain_X.append(X_c)
    all_pretrain_y.append(y_c)

X_pretrain = np.concatenate(all_pretrain_X, axis=0)
y_pretrain = np.concatenate(all_pretrain_y, axis=0)

print("Combined pretraining data shape:", X_pretrain.shape)
print("Label distribution:", dict(zip(*np.unique(y_pretrain, return_counts=True))))

Combined pretraining data shape: (25608, 22, 500)
Label distribution: {np.int64(0): np.int64(6292), np.int64(1): np.int64(6501), np.int64(2): np.int64(6314), np.int64(3): np.int64(6501)}


In [20]:
from braindecode.models import Deep4Net

# Split combined data into pretrain-train/val
n_total = len(X_pretrain)
rng = np.random.RandomState(RANDOM_SEED)
idx = rng.permutation(n_total)
n_val = int(0.1 * n_total)
val_idx, train_idx = idx[:n_val], idx[n_val:]

# Normalize (fit on pretrain-train only)
pretrain_mean = X_pre_tr.mean(axis=(0, 2), keepdims=True)
pretrain_std = X_pre_tr.std(axis=(0, 2), keepdims=True)

X_pre_tr_norm = (X_pre_tr - pretrain_mean) / (pretrain_std + 1e-8)
X_pre_val_norm = (X_pre_val - pretrain_mean) / (pretrain_std + 1e-8)

pretrain_loader = DataLoader(EEGDataset(X_pre_tr_norm, y_pre_tr), batch_size=128, shuffle=True)
pretrain_val_loader = DataLoader(EEGDataset(X_pre_val_norm, y_pre_val), batch_size=128, shuffle=False)

print(f"Pretraining on {len(X_pre_tr)} samples, validating on {len(X_pre_val)}")

# Initialize Deep4Net instead of EEGConformer
pretrained_model = Deep4Net(
    n_chans=22, 
    n_outputs=4, 
    n_times=crop_len, 
    drop_prob=0.5
)

print("Running Cross-Subject Pretraining with Deep4Net (up to 100 epochs)...")

# Keeping Mixup OFF for preprocessed data
pretrained_model, pretrain_hist = train_model(
    pretrained_model,
    pretrain_loader,
    pretrain_val_loader,
    n_epochs=100,
    lr=0.001,
    patience=15,
    weight_decay=0.01,
    verbose=True,
    apply_mixup=False 
)

print("\nPretraining complete.")

Pretraining on 23048 samples, validating on 2560
Running Cross-Subject Pretraining with Deep4Net (up to 100 epochs)...
Epoch 1/100 | Train Loss: 1.4871 Acc: 0.2744 | Val Loss: 1.3625 Acc: 0.3187
Epoch 11/100 | Train Loss: 1.1955 Acc: 0.5060 | Val Loss: 1.1616 Acc: 0.5355
Epoch 21/100 | Train Loss: 1.1623 Acc: 0.5254 | Val Loss: 1.1374 Acc: 0.5410
Epoch 31/100 | Train Loss: 1.1464 Acc: 0.5396 | Val Loss: 1.0763 Acc: 0.6082
Epoch 41/100 | Train Loss: 1.1226 Acc: 0.5565 | Val Loss: 1.0847 Acc: 0.5668
Epoch 51/100 | Train Loss: 1.0920 Acc: 0.5792 | Val Loss: 1.0167 Acc: 0.6379
Epoch 61/100 | Train Loss: 1.0744 Acc: 0.5919 | Val Loss: 0.9999 Acc: 0.6652
Epoch 71/100 | Train Loss: 1.0454 Acc: 0.6102 | Val Loss: 0.9720 Acc: 0.6895
Epoch 81/100 | Train Loss: 1.0222 Acc: 0.6284 | Val Loss: 0.9494 Acc: 0.6988
Epoch 91/100 | Train Loss: 1.0118 Acc: 0.6331 | Val Loss: 0.9323 Acc: 0.7180
Epoch 100/100 | Train Loss: 1.0003 Acc: 0.6450 | Val Loss: 0.9282 Acc: 0.7207

Pretraining complete.


In [24]:
import copy as copy_module
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score
from braindecode.models import Deep4Net
import torch
from torch.utils.data import DataLoader

# --- ADDED: Euclidean Alignment Function ---
def euclidean_alignment(X):
    # X shape: (n_trials, n_channels, n_times)
    n_trials, n_channels, n_times = X.shape
    # Compute covariance for each trial
    covs = np.array([np.cov(trial) for trial in X])
    # Mean covariance across all trials
    mean_cov = np.mean(covs, axis=0)
    # Inverse square root of mean_cov
    evals, evecs = np.linalg.eigh(mean_cov)
    evals = np.maximum(evals, 1e-10)
    inv_sqrt_mean_cov = evecs @ np.diag(evals**(-0.5)) @ evecs.T
    
    # Align trials
    return np.array([inv_sqrt_mean_cov @ trial for trial in X])

def finetune_subject(subject_id, pretrained_state_dict, crop_len=500, stride=50,
                       n_epochs=50, lr=0.0001, patience=15,
                       weight_decay=0.01, drop_prob=0.5, batch_size=32, val_frac=0.2, seed=42):
    
    X_train_full = all_epochs[f"A{subject_id}T"].get_data()
    y_train_full = all_labels[f"A{subject_id}T"] - 1
    X_test_full = all_epochs[f"A{subject_id}E"].get_data()
    y_test_full = all_labels[f"A{subject_id}E"] - 1

    # Trial-level split, then crop
    n_trials = len(X_train_full)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n_trials)
    n_val = int(val_frac * n_trials)
    val_idx, train_idx = indices[:n_val], indices[n_val:]

    X_tr_trials, y_tr_trials = X_train_full[train_idx], y_train_full[train_idx]
    X_val_trials, y_val_trials = X_train_full[val_idx], y_train_full[val_idx]

    X_tr_c, y_tr_c, _ = crop_array(X_tr_trials, y_tr_trials, crop_len, stride)
    X_val_c, y_val_c, _ = crop_array(X_val_trials, y_val_trials, crop_len, stride)

    # --- APPLY EUCLIDEAN ALIGNMENT ---
    X_tr_c = euclidean_alignment(X_tr_c)
    X_val_c = euclidean_alignment(X_val_c)

    # Normalization
    X_tr_norm = (X_tr_c - pretrain_mean) / (pretrain_std + 1e-8)
    X_val_norm = (X_val_c - pretrain_mean) / (pretrain_std + 1e-8)

    train_loader = DataLoader(EEGDataset(X_tr_norm, y_tr_c), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val_norm, y_val_c), batch_size=batch_size, shuffle=False)

    model = Deep4Net(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=drop_prob)
    model.load_state_dict(copy_module.deepcopy(pretrained_state_dict))

    finetuned, hist = train_model(model, train_loader, val_loader, n_epochs=n_epochs,
                                    lr=lr, patience=patience, weight_decay=weight_decay, verbose=False, apply_mixup=False)

    # Evaluate on E-session: Align -> Normalize -> Majority Vote
    X_test_c, y_test_c, test_trial_ids = crop_array(X_test_full, y_test_full, crop_len, stride)
    
    # Align test data using the SAME alignment logic
    X_test_c = euclidean_alignment(X_test_c)
    X_test_norm = (X_test_c - pretrain_mean) / (pretrain_std + 1e-8)

    test_loader = DataLoader(EEGDataset(X_test_norm, y_test_c), batch_size=batch_size, shuffle=False)

    finetuned.eval()
    crop_preds = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device)
            out = finetuned(Xb)
            crop_preds.extend(out.argmax(1).cpu().numpy())
    crop_preds = np.array(crop_preds)

    n_test_trials = len(X_test_full)
    final_preds = []
    for trial_i in range(n_test_trials):
        mask = test_trial_ids == trial_i
        votes = crop_preds[mask]
        majority = np.bincount(votes).argmax()
        final_preds.append(majority)
    final_preds = np.array(final_preds)

    test_acc = accuracy_score(y_test_full, final_preds)
    test_kappa = cohen_kappa_score(y_test_full, final_preds)
    best_val_acc = max(hist['val_acc'])

    print(f"Subject A{subject_id}: best_val_acc={best_val_acc:.4f} | test_acc={test_acc:.4f} | test_kappa={test_kappa:.4f}")

    return {'subject': f"A{subject_id}", 'best_val_acc': best_val_acc,
            'test_acc': test_acc, 'test_kappa': test_kappa}

# Run fine-tuning
pretrained_state = pretrained_model.state_dict()
results_finetuned = []
print("Running Fine-Tuning with Euclidean Alignment (Deep4Net)...\n")
for subj in subjects:
    result = finetune_subject(subj, pretrained_state)
    results_finetuned.append(result)

results_finetuned_df = pd.DataFrame(results_finetuned)
print("\n=== Final Summary (Deep4Net + Euclidean Alignment) ===")
print(results_finetuned_df)
print(f"\nMean test accuracy: {results_finetuned_df['test_acc'].mean():.4f}")
print(f"Mean test kappa: {results_finetuned_df['test_kappa'].mean():.4f}")

Running Fine-Tuning with Euclidean Alignment (Deep4Net)...

Subject A01: best_val_acc=0.8552 | test_acc=0.8577 | test_kappa=0.8103
Subject A02: best_val_acc=0.6364 | test_acc=0.5654 | test_kappa=0.4212
Subject A03: best_val_acc=0.8316 | test_acc=0.8242 | test_kappa=0.7653
Subject A04: best_val_acc=0.7570 | test_acc=0.6316 | test_kappa=0.5084
Subject A05: best_val_acc=0.7045 | test_acc=0.5471 | test_kappa=0.3984
Subject A06: best_val_acc=0.6660 | test_acc=0.5814 | test_kappa=0.4429
Subject A07: best_val_acc=0.8670 | test_acc=0.8303 | test_kappa=0.7738
Subject A08: best_val_acc=0.8339 | test_acc=0.7454 | test_kappa=0.6611
Subject A09: best_val_acc=0.8162 | test_acc=0.7765 | test_kappa=0.7020

=== Final Summary (Deep4Net + Euclidean Alignment) ===
  subject  best_val_acc  test_acc  test_kappa
0     A01      0.855219  0.857651    0.810286
1     A02      0.636364  0.565371    0.421172
2     A03      0.831650  0.824176    0.765330
3     A04      0.756993  0.631579    0.508368
4     A05      